# T2.3 – Mapping Units of Measurement

## Vienna Weather Wet-Month Prediction Experiment

This notebook assigns ontology-based unit mappings to all numeric attributes
in the DBRepo schema.

The mappings support the FAIR principles by improving:
- semantic interoperability,
- machine readability,
- metadata quality,
- and reusability of the experiment data.

## Ontology Choice

The recommended ontology for this task was the SI Digital Framework.
For practical integration with the DBRepo test instance, the OM-2 ontology
(Ontology of Units of Measure) was selected because it is already registered
and supported within the DBRepo metadata registry.

OM-2 concepts are used for:
- temperature
- atmospheric pressure
- precipitation
- wind speed
- humidity
- temporal units
- spatial coordinates
- dimensionless identifiers
- count-based quantities

## Dataset

Source dataset:
Stadt Wien – Monthly weather observations at Hohe Warte station since 1872.

License:
CC BY 4.0

## 1. Import libraries and configure DBRepo connection

We connect to the DBRepo instance using the Python REST client.
The database and table identifiers were created previously in T2.1.

In [45]:
import pandas as pd
import requests
from dbrepo.RestClient import RestClient

In [46]:
ENDPOINT = "https://test.dbrepo.tuwien.ac.at"

USERNAME = "azra1558"
PASSWORD = "Katalizator1558!"

DATABASE_ID = "899bfcba-7fec-40c9-9076-3a3a9372c844"

TABLE_IDS = {
    "weather_measurement": "2212bed4-ef8f-4d95-bb65-20b2adb28abd",
    "time_dimension": "9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2",
    "station": "e6779029-ce40-4a9a-ad17-147e183dc757"
}

client = RestClient(
    endpoint=ENDPOINT,
    username=USERNAME,
    password=PASSWORD
)

print("Connected to DBRepo.")

Connected to DBRepo.


## 2. Load the unit mapping table

The ontology mappings are maintained in a CSV file to ensure:
- transparency,
- reproducibility,
- and easier maintenance of semantic metadata.

Each row contains:
- table name,
- column name,
- ontology URI,
- human-readable unit label.

In [47]:
mapping_df = pd.read_csv("../docs/unit_mapping.csv")

mapping_df.head()

,table_name,column_name,unit_uri,unit_label
0,weather_measurement,measurement_id,http://www.ontology-of-units-of-measure.org/re...,unitless
1,weather_measurement,station_num,http://www.ontology-of-units-of-measure.org/re...,unitless
2,weather_measurement,time_id,http://www.ontology-of-units-of-measure.org/re...,unitless
3,weather_measurement,t_mean_c,http://www.ontology-of-units-of-measure.org/re...,degree Celsius
4,weather_measurement,t_max_c,http://www.ontology-of-units-of-measure.org/re...,degree Celsius


## 3. Validate DB schema

Before assigning ontology mappings, we verify that all referenced
tables and columns exist in the DBRepo schema.

In [48]:
tables = client.get_tables(DATABASE_ID)

for t in tables:
    print(t.name, t.id)

weather_measurement 2212bed4-ef8f-4d95-bb65-20b2adb28abd
time_dimension 9f4dc236-81cc-4bdb-93e7-a8b6ed5436f2
station e6779029-ce40-4a9a-ad17-147e183dc757


## 4. Validate unit mappings

This step validates that:
- all mapped tables exist,
- all mapped columns exist,
- and every numeric attribute has a corresponding ontology-based unit mapping.

In [49]:
success = 0
failed = 0

for _, row in mapping_df.iterrows():

    table_name = row["table_name"]
    column_name = row["column_name"]
    unit_uri = row["unit_uri"]

    table_id = TABLE_IDS[table_name]

    try:

        table = client.get_table(DATABASE_ID, table_id)

        col = next(
            (c for c in table.columns if c.name == column_name),
            None
        )

        if col is None:
            print(f"FAILED: {table_name}.{column_name} not found")
            failed += 1

        else:
            print(
                f"OK: {table_name}.{column_name} "
                f"→ {unit_uri}"
            )
            success += 1

    except Exception as e:
        print(f"FAILED ({e}): {table_name}.{column_name}")
        failed += 1

print("\n===================================")
print(f"Validated mappings: {success}")
print(f"Failed mappings   : {failed}")
print("===================================")

OK: weather_measurement.measurement_id → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement.station_num → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement.time_id → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement.t_mean_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.t_max_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.t_min_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.mean_t_max_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.mean_t_min_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.p_mean_hpa → http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal
OK: weather_measurement.p_max_hpa → http://www.ontolo

## 5. Attempt DBRepo metadata integration

As required by the assignment, we attempt to assign ontology-based
unit mappings directly to DBRepo columns using the REST API.

In [53]:
success = 0
failed = 0

for _, row in mapping_df.iterrows():
    table_name = row["table_name"]
    column_name = row["column_name"]
    unit_uri = row["unit_uri"]
    table_id = TABLE_IDS[table_name]

    table = client.get_table(DATABASE_ID, table_id)
    col = next((c for c in table.columns if c.name == column_name), None)

    if col is None:
        print(f"FAILED (not found): {table_name}.{column_name}")
        failed += 1
        continue

    url = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{table_id}/column/{col.id}"
    r = requests.put(url, json={"unit_uri": unit_uri}, auth=(USERNAME, PASSWORD))

    if r.status_code in (200, 202, 204):
        print(f"OK: {table_name}.{column_name} → {unit_uri}")
        success += 1
    else:
        print(f"FAILED (status {r.status_code}): {table_name}.{column_name}")
        failed += 1

print(f"\nUpload success : {success}")
print(f"Upload failed  : {failed}")

OK: weather_measurement.measurement_id → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement.station_num → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement.time_id → http://www.ontology-of-units-of-measure.org/resource/om-2/one
OK: weather_measurement.t_mean_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.t_max_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.t_min_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.mean_t_max_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.mean_t_min_c → http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius
OK: weather_measurement.p_mean_hpa → http://www.ontology-of-units-of-measure.org/resource/om-2/hectopascal
OK: weather_measurement.p_max_hpa → http://www.ontolo

In [54]:
table_after = client.get_table(DATABASE_ID, TABLE_IDS["weather_measurement"])
col_after = next(c for c in table_after.columns if c.name == "t_mean_c")
print("Unit:", col_after.unit)

Unit: None


In [55]:
# Single test with full debug
table = client.get_table(DATABASE_ID, TABLE_IDS["weather_measurement"])
col = next(c for c in table.columns if c.name == "t_mean_c")
print("Column ID:", col.id)

url = f"{ENDPOINT}/api/v1/database/{DATABASE_ID}/table/{TABLE_IDS['weather_measurement']}/column/{col.id}"
r = requests.put(url, json={"unit_uri": "http://www.ontology-of-units-of-measure.org/resource/om-2/degreeCelsius"}, auth=(USERNAME, PASSWORD))
print("Status:", r.status_code)
print("Response:", r.text[:500])

# Immediately verify
import time
time.sleep(2)
table2 = client.get_table(DATABASE_ID, TABLE_IDS["weather_measurement"])
col2 = next(c for c in table2.columns if c.name == "t_mean_c")
print("Unit after update:", col2.unit)

Column ID: 6d030a9e-4986-4ba5-9d26-9ad80ae3c9ed
Status: 202
Response: 
Unit after update: None
